In [1]:
import torch
from mini_whisper import *
from mini_whisper.transformer.MHA_simple import SimpleTransformerBlock
from mini_whisper.transformer.MHA import TransformerBlock

In [2]:
SPLIT = "dev-clean"
DATA_DIR = "./data"
FOLDER_IN_ARCHIVE = "LibriSpeech"
BATCH_SIZE = 16
N_MELS = 80
D_MODEL = 128
N_HEADS = 8

print("=" * 60)
print("Mini-Whisper Training - Data Loading & Preprocessing")
print("=" * 60)


Mini-Whisper Training - Data Loading & Preprocessing


In [3]:
print(f"\nCreating DataLoader for: {DATA_DIR}")
print(f"Batch size: {BATCH_SIZE}")

dataloader = LibriSpeechAudioPreprocessingDataLoader(
    split=SPLIT,
    root_dir=DATA_DIR,
    folder_in_archive=FOLDER_IN_ARCHIVE,
    download_dataset=True,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    n_mel_bins=N_MELS
)

print(f"\nDataLoader created with {len(dataloader.dataset)} samples")
print(f"  Number of batches: {len(dataloader)}")



Creating DataLoader for: ./data
Batch size: 16

DataLoader created with 2703 samples
  Number of batches: 169


In [4]:
print(f"\nInitializing AudioEncoderStem (n_mels={N_MELS}, d_model={D_MODEL})")
stem = AudioEncoderStem(n_mels=N_MELS, d_model=D_MODEL)
stem.eval()
print("Encoder stem initialized")



Initializing AudioEncoderStem (n_mels=80, d_model=128)
Encoder stem initialized


In [5]:
print(f"\nProcessing first batch...")
batch = next(iter(dataloader))

log_mels = batch['log_mel']  # (B, n_mels, T)
transcripts = batch['transcript']
audio_paths = batch['audio_path']

print(f"\nBatch contents:")
print(f"  Log-mel shape: {log_mels.shape}")
print(f"  Number of transcripts: {len(transcripts)}")



Processing first batch...

Batch contents:
  Log-mel shape: torch.Size([16, 80, 3000])
  Number of transcripts: 16


In [6]:
print(f"\nFirst 3 samples in batch:")
for i in range(min(3, len(transcripts))):
    print(f"  {i+1}. {audio_paths[i]}")
    print(f"     Transcript: {transcripts[i][:60]}...")



First 3 samples in batch:
  1. 8842/304647/8842-304647-0004.flac
     Transcript: MONEY IS FALSE AND LIGHT UNLESS IT BE BOUGHT BY A MAN'S OWN ...
  2. 3752/4944/3752-4944-0050.flac
     Transcript: YOU'RE A DISMISSED OFFICER OF THE GOVERNMENT SIR...
  3. 5895/34615/5895-34615-0008.flac
     Transcript: THE OUTSIDE DID NOT DEPEND ON THE INTERIOR...


In [7]:
# Let's first start just with the stem
with torch.no_grad():
    batch_features = stem(log_mels)

print(f"\nEncoder output shape: {batch_features.shape}")



Encoder output shape: torch.Size([16, 1500, 128])


In [8]:
# Now let us add a layer of the encoder self-attention transformer block
print(f"\nInitializing AudioEncoderLayer (d_model={D_MODEL})")
encoder_layer = SimpleTransformerBlock(d_model=D_MODEL, n_head=N_HEADS)
with torch.no_grad():
    x = stem(log_mels)
    z = encoder_layer(x)
print(f"Output shape after encoder layer: {batch_features.shape}")



Initializing AudioEncoderLayer (d_model=128)
Output shape after encoder layer: torch.Size([16, 1500, 128])


In [9]:
# Okay, time for a encoder+decoder pair
print(f"\nInitializing AudioEncoderLayer (d_model={D_MODEL})")
encoder_layer = SimpleTransformerBlock(d_model=D_MODEL, n_head=N_HEADS)
decoder_layer = TransformerBlock(d_model=D_MODEL, n_head=N_HEADS, cross_attention=True)
with torch.no_grad():
    x = stem(log_mels)
    z = encoder_layer(x)
    y = decoder_layer(z, z)  # Using encoder output as both input and context for testing
print(f"Output shape after encoder+decoder layer: {y.shape}")


Initializing AudioEncoderLayer (d_model=128)
Output shape after encoder+decoder layer: torch.Size([16, 1500, 128])


In [10]:
# And now to decode the results
from mini_whisper.tokenizer.tokenizer import BPE_Tokenizer
from mini_whisper.decoder.textDecoder import TextDecoder
from torch.nn import functional as F

tokenizer = BPE_Tokenizer()
tokenizer.load_merges('mini_whisper/decoder/merges.txt')
seqs = [tokenizer.encode("hello world"),
        tokenizer.encode("this is a test")]          

txt = torch.tensor(seqs, dtype=torch.long)          
B, T = txt.shape                                    
max_len = 1500

# 1) Pad last dim from 3 -> 20 with zeros
pad_len = max_len - T
txt_padded = F.pad(txt, (0, pad_len), value=0)      # [2, 20]

# 2) Expand batch dim from 2 -> 16 by repeating
repeats = (BATCH_SIZE + B - 1) // B               # ceil(16 / 2) = 8
txt_big = txt_padded.repeat(repeats, 1)[:BATCH_SIZE]  # [16, 20]

decoder = TextDecoder(vocab_size=10000, d_model=D_MODEL, max_len=1500, n_layers=3, n_heads=4)
with torch.no_grad():
        print(f'x: {txt_big.shape}, z: {z.shape}')  # [16, seq_len, d_model]
        logits = decoder(txt_big, z)
        print(f'logits shape: {logits.shape}')  # Should be [16, seq_len, vocab_size]
        print(F.softmax(logits, dim=-1).shape)  # Should also be [16, seq_len, vocab_size]
        print(F.softmax(logits, dim=-1)[0, 0, :10])  # Print probabilities of first 10 tokens for the first position in the first batch item
        greedy_token = torch.argmax(F.softmax(logits, dim=-1), dim=-1)  # [16, seq_len]
        print(f'greedy token shape: {greedy_token.shape}')  # Should be [16, seq_len]
        print(f'greedy token for first position in first batch item: {greedy_token[0, 0]}')
        list(map(lambda i: print(f'greedy token for position {i} in first batch item: {tokenizer.decode([greedy_token[0, i].item()])}'), range(20)))

x: torch.Size([16, 1500]), z: torch.Size([16, 1500, 128])
pos emb shape: torch.Size([16, 1500, 128]), token shape: torch.Size([16, 1500, 128])
logits shape: torch.Size([16, 1500, 10000])
torch.Size([16, 1500, 10000])
tensor([2.0279e-13, 3.7307e-23, 4.1832e-16, 2.3539e-24, 1.7838e-16, 1.4093e-25,
        9.5708e-21, 6.1317e-20, 3.8505e-25, 1.2472e-09])
greedy token shape: torch.Size([16, 1500])
greedy token for first position in first batch item: 5270
greedy token for position 0 in first batch item: fter 
greedy token for position 1 in first batch item: fter 
greedy token for position 2 in first batch item: fter 
greedy token for position 3 in first batch item: fter 
greedy token for position 4 in first batch item: fter 
greedy token for position 5 in first batch item: fter 
greedy token for position 6 in first batch item: fter 
greedy token for position 7 in first batch item: fter 
greedy token for position 8 in first batch item: fter 
greedy token for position 9 in first batch item: f